In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# 1) Create a very small DataFrame
data = [
    ("Product A", "2026-01-01", 100),
    ("Product A", "2026-02-01", 150),
    ("Product A", "2026-03-01", 120),
    ("Product A", "2026-04-01", 180),
    ("Product A", "2026-05-01", 200),
]
df = spark.createDataFrame(data, ["product", "month", "sales"]) \
         .withColumn("month", F.to_date("month"))

# 2) Define the window: compare rows within the same product and ordered by month
w = Window.partitionBy("product").orderBy("month")

# 3) Build the features using LAG, LEAD, and ROWS BETWEEN
result = (
    df
    # previous month sales (LAG)
    .withColumn("prev_month_sales", F.lag("sales", 1).over(w))
    # next month sales (LEAD)
    .withColumn("next_month_sales", F.lead("sales", 1).over(w))
    # month-on-month % change vs previous month
    .withColumn(
        "mom_change_pct",
        F.round(
            F.when(F.col("prev_month_sales") > 0,
                   (F.col("sales") - F.col("prev_month_sales")) / F.col("prev_month_sales") * 100
            ),
            2
        )
    )
    # rolling sum of current + previous 2 rows (i.e., last 3 months) using ROWS BETWEEN
    .withColumn(
        "rolling_3_month_sum",
        F.sum("sales").over(w.rowsBetween(-2, 0))
    )
    .orderBy("product", "month")
)

display(result)
